Estimators and Non-parametric Models

Key concepts:

- Point estimation
- Confidence intervals
- Empirical distribution function

Equations:

Sample mean:
$$[ \bar{x} = \frac{1}{n} \sum_{i=1}^n x_i ]$$

Sample variance:

$$[ s^2 = \frac{1}{n-1} \sum_{i=1}^n (x_i - \bar{x})^2 ]$$

Confidence interval for mean (normal distribution):
$$[ CI = \bar{x} \pm t_{\alpha/2, n-1} \cdot \frac{s}{\sqrt{n}} ]$$




In [1]:
import numpy as np
from scipy import stats

def sample_mean(data):
    return np.mean(data)

def sample_variance(data):
    return np.var(data, ddof=1)

def confidence_interval(data, confidence=0.95):
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    ci = stats.t.interval(confidence, n-1, loc=mean, scale=se)
    return ci

## Test de Hipotesis
Key concepts:

- Null and alternative hypotheses
- Type I and Type II errors
- p-value
- Test statistics (z-test, t-test, chi-square test)


Z-statistic:
$$[ z = \frac{\bar{x} - \mu_0}{\sigma / \sqrt{n}} ]$$

T-statistic:
$$[ t = \frac{\bar{x} - \mu_0}{s / \sqrt{n}} ]$$


In [ ]:
def z_test(data, null_mean, known_std_dev, alpha=0.05):
    sample_mean = np.mean(data)
    n = len(data)
    z_stat = (sample_mean - null_mean) / (known_std_dev / np.sqrt(n))
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))
    return z_stat, p_value

def t_test(data, null_mean, alpha=0.05):
    t_stat, p_value = stats.ttest_1samp(data, null_mean)
    return t_stat, p_value

## 3 Linear Regression

Key concepts:

- Ordinary Least Squares (OLS) AKA Minimos Cuadrados Ordinarios MCE
- Coefficient of determination (R²)
- Residuals analysis

Equations:
Slope estimator:
$$[ \hat{\beta}1 = \frac{\sum_{i=1}^n (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^n (x_i - \bar{x})^2} ]$$
Intercept estimator:
$$[ \hat{\beta}_0 = \bar{y} - \hat{\beta}_1 \bar{x} ]$$
Coefficient of determination:
$$[ R^2 = 1 - \frac{SS_{res}}{SS_{tot}} = 1 - \frac{\sum_{i=1}^n (y_i - \hat{y}i)^2}{\sum_{i=1}^n (y_i - \bar{y})^2} ]$$

In [ ]:
def linear_regression(X, y):
    X = np.column_stack((np.ones(len(X)), X))
    beta = np.linalg.inv(X.T @ X) @ X.T @ y
    return beta

def r_squared(y, y_pred):
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - np.mean(y))**2)
    return 1 - (ss_res / ss_tot)

## 4 Generalized Linear Models (GLM)

Key concepts:

- Exponential dispersion models
- Link functions
- Maximum likelihood estimation

Equations:

Log-likelihood for Poisson GLM:
$$[ l(\beta) = \sum_{i=1}^n \left(y_i (x_i^T \beta) - \exp(x_i^T \beta) - \log(y_i!) \right) ]$$
Link function for Poisson GLM:
$$[ g(\mu) = ]$$

In [ ]:
from scipy.optimize import minimize

def poisson_glm(X, y):
    def negative_log_likelihood(beta):
        mu = np.exp(X @ beta)
        return -np.sum(y * np.log(mu) - mu - np.log(np.factorial(y)))
    
    initial_beta = np.zeros(X.shape[1])
    result = minimize(negative_log_likelihood, initial_beta, method='BFGS')
    return result.x

## 5 Bootstrap

Key concepts:

- Resampling with replacement
- Bootstrap confidence intervals
- Standard error estimation

1. **Bootstrap Estimate of Standard Error**:
   $$ 
   SE_B(\hat{\theta}) = \sqrt{\frac{1}{B-1} \sum_{b=1}^B \left( \hat{\theta}^{(b)} - \bar{\theta}^* \right)^2} 
   $$

   Where:
   $$ 
   \bar{\theta}^* = \frac{1}{B} \sum_{b=1}^B \hat{\theta}^{(b)} 
   $$

2. **Bootstrap Percentile Confidence Interval**:
   $$ 
   CI = \left[ \hat{\theta}_{(\alpha/2)}, \hat{\theta}_{(1-\alpha/2)} \right] 
   $$

Python code: 

In [ ]:
def bootstrap_statistic(data, statistic_func, num_samples=1000):
    bootstrap_stats = []
    for _ in range(num_samples):
        resample = np.random.choice(data, size=len(data), replace=True)
        bootstrap_stats.append(statistic_func(resample))
    return np.array(bootstrap_stats)

def bootstrap_ci(data, statistic_func, alpha=0.05, num_samples=1000):
    bootstrap_stats = bootstrap_statistic(data, statistic_func, num_samples)
    ci_lower = np.percentile(bootstrap_stats, alpha/2 * 100)
    ci_upper = np.percentile(bootstrap_stats, (1 - alpha/2) * 100)
    return ci_lower, ci_upper

### Ejercicio 1
Consideramos la distribución gaussiana inversa:
$$f(x; \mu, \phi) = \frac{1}{\sqrt{2\pi x^3 \phi}} e^{-\frac{(x-\mu)^2}{2\phi x \mu^2}}$$
a) Estimadores de momentos de μ y φ
Para encontrar los estimadores de momentos, necesitamos calcular la esperanza y la varianza de la distribución.
Para la gaussiana inversa:

$E[X] = \mu$
$Var(X) = \mu^3 / \lambda = \mu^3 \phi$

Los estimadores de momentos serían:
$$\hat{\mu} = \bar{X} = \frac{1}{n}\sum_{i=1}^n X_i$$
$$\hat{\phi} = \frac{S^2}{\bar{X}^3}$$
donde $S^2 = \frac{1}{n-1}\sum_{i=1}^n (X_i - \bar{X})^2$ es la varianza muestral.
b) Estimadores de máxima verosimilitud de μ y φ
Para encontrar los estimadores de máxima verosimilitud, maximizamos la función de log-verosimilitud:
$$\ell(\mu, \phi) = -\frac{n}{2}\log(2\pi\phi) - \frac{3}{2}\sum_{i=1}^n \log(x_i) - \frac{1}{2\phi\mu^2}\sum_{i=1}^n \frac{(x_i - \mu)^2}{x_i}$$
Derivando respecto a μ y φ e igualando a cero:
$$\hat{\mu}_{MV} = \bar{X}$$
$$\hat{\phi}{MV} = \frac{1}{n}\sum{i=1}^n \frac{(X_i - \hat{\mu}{MV})^2}{X_i \hat{\mu}{MV}^2}$$
c) Insesgadez y error cuadrático medio de los estimadores de μ
El estimador de momentos y de máxima verosimilitud para μ es el mismo: $\hat{\mu} = \bar{X}$.
Este estimador es insesgado, ya que $E[\hat{\mu}] = E[\bar{X}] = \mu$.
El error cuadrático medio (ECM) es:
$$ECM(\hat{\mu}) = Var(\hat{\mu}) + (E[\hat{\mu}] - \mu)^2 = Var(\bar{X}) + 0 = \frac{Var(X)}{n} = \frac{\mu^3\phi}{n}$$
d) Afirmación sobre estimadores insesgados vs sesgados
La afirmación "Dados dos estimadores de un parámetro θ, siempre conviene usar un estimador insesgado por sobre uno sesgado" es falsa.
Justificación: Aunque la insesgadez es una propiedad deseable, no es el único criterio para elegir un estimador. El error cuadrático medio (ECM) es una medida más completa de la calidad de un estimador, ya que considera tanto el sesgo como la varianza. En algunos casos, un estimador sesgado puede tener un ECM menor que un estimador insesgado, lo que lo haría preferible en términos de precisión global.